In [2]:
from pathlib import Path

import numpy as np
import xarray as xr
import rioxarray


# ============================================================
# PATHS
# ============================================================

AVG_FILE = Path("data_stream-oper_stepType-avg.nc")
INSTANT_FILE = Path("data_stream-oper_stepType-instant.nc")
MAX_FILE = Path("data_stream-oper_stepType-max.nc")

OUTPUT_DIR = Path("derived/era5_icing")
RASTER_DIR = OUTPUT_DIR / "rasters"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RASTER_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# HELPERS
# ============================================================

def normalize_dataset(ds):
    """
    Normalize coordinate names and longitude convention.
    """

    rename = {}

    if "time" in ds.dims and "valid_time" not in ds.dims:
        rename["time"] = "valid_time"

    if "lat" in ds.dims and "latitude" not in ds.dims:
        rename["lat"] = "latitude"

    if "lon" in ds.dims and "longitude" not in ds.dims:
        rename["lon"] = "longitude"

    if rename:
        ds = ds.rename(rename)

    # Convert 0..360 longitude convention to -180..180.
    if float(ds.longitude.max()) > 180:
        ds = ds.assign_coords(
            longitude=((ds.longitude + 180) % 360) - 180
        ).sortby("longitude")

    return ds


def align_to_reference(ds, reference):
    """
    Verify grids are physically equivalent, then adopt the
    reference dataset's exact coordinate labels.
    """

    assert ds.sizes["valid_time"] == reference.sizes["valid_time"]
    assert ds.sizes["latitude"] == reference.sizes["latitude"]
    assert ds.sizes["longitude"] == reference.sizes["longitude"]

    assert np.array_equal(
        ds.valid_time.values,
        reference.valid_time.values,
    )

    assert np.allclose(
        ds.latitude.values,
        reference.latitude.values,
    )

    assert np.allclose(
        ds.longitude.values,
        reference.longitude.values,
    )

    return ds.assign_coords(
        valid_time=reference.valid_time,
        latitude=reference.latitude,
        longitude=reference.longitude,
    )


def export_geotiff(data_array, filename):
    """
    Export a two-dimensional ERA5-derived result as EPSG:4326.
    QGIS can reproject it on the fly to EPSG:5070.
    """

    da = data_array.copy()

    if "quantile" in da.coords:
        da = da.drop_vars("quantile")

    da = da.rio.set_spatial_dims(
        x_dim="longitude",
        y_dim="latitude",
    )

    da = da.rio.write_crs("EPSG:4326")

    output_path = RASTER_DIR / filename

    da.rio.to_raster(
        output_path,
        compress="DEFLATE",
    )

    print(f"Saved: {output_path}")


# ============================================================
# 1. LOAD FILES
# ============================================================

avg = normalize_dataset(xr.open_dataset(AVG_FILE))
instant = normalize_dataset(xr.open_dataset(INSTANT_FILE))
maximum = normalize_dataset(xr.open_dataset(MAX_FILE))

print("AVG variables:", list(avg.data_vars))
print("INSTANT variables:", list(instant.data_vars))
print("MAX variables:", list(maximum.data_vars))


# ============================================================
# 2. ALIGN + MERGE
# ============================================================

# Use the instantaneous product as the canonical grid.
avg = align_to_reference(avg, instant)
maximum = align_to_reference(maximum, instant)

era5 = xr.merge(
    [instant, avg, maximum],
    join="exact",
    compat="no_conflicts",
)

print("\nMerged dimensions:")
print(era5.sizes)

print("\nMerged variables:")
print(list(era5.data_vars))


# ============================================================
# 3. BASIC UNIT CONVERSIONS / DERIVED VARIABLES
# ============================================================

# Air temperature: Kelvin -> Celsius
era5["t2m_c"] = era5["t2m"] - 273.15
era5["t2m_c"].attrs = {
    "long_name": "2 metre air temperature",
    "units": "degC",
}

# Dewpoint: Kelvin -> Celsius
era5["d2m_c"] = era5["d2m"] - 273.15
era5["d2m_c"].attrs = {
    "long_name": "2 metre dewpoint temperature",
    "units": "degC",
}

# Dewpoint depression:
# small values indicate air close to saturation
era5["dewpoint_depression_c"] = (
    era5["t2m_c"] - era5["d2m_c"]
)

era5["dewpoint_depression_c"].attrs = {
    "long_name": "2 metre dewpoint depression",
    "units": "degC",
}


# ------------------------------------------------------------
# Precipitation rates
#
# kg m^-2 s^-1 -> mm/hour
#
# 1 kg/m² liquid water = 1 mm water depth.
# ------------------------------------------------------------

era5["precip_rate_mmh"] = era5["avg_tprate"] * 3600

era5["precip_rate_mmh"].attrs = {
    "long_name": "Mean total precipitation rate",
    "units": "mm h-1",
}


era5["snowfall_rate_mmh"] = era5["avg_tsrwe"] * 3600

era5["snowfall_rate_mmh"].attrs = {
    "long_name": "Mean snowfall water-equivalent rate",
    "units": "mm h-1 water equivalent",
}


# ------------------------------------------------------------
# Low cloud cover
#
# ERA5 normally stores this as a 0..1 fraction.
# Convert to percentage if so.
# ------------------------------------------------------------

if float(era5["lcc"].max()) <= 1.5:

    era5["low_cloud_cover_pct"] = (
        era5["lcc"] * 100
    )

else:

    era5["low_cloud_cover_pct"] = era5["lcc"]


era5["low_cloud_cover_pct"].attrs = {
    "long_name": "Low cloud cover",
    "units": "%",
}


# ============================================================
# 4. TEMPERATURE / FREEZING METRICS
# ============================================================

mean_temp = era5["t2m_c"].mean(
    dim="valid_time"
)

mean_temp.name = "mean_temperature"


freezing_pct = (
    (era5["t2m_c"] < 0)
    .mean(dim="valid_time")
    * 100
)

freezing_pct.name = "freezing_hours_pct"

freezing_pct.attrs = {
    "long_name": (
        "Percentage of January hours "
        "with 2 metre temperature below 0 degC"
    ),
    "units": "%",
}


# ============================================================
# 5. HUMIDITY / SATURATION CONTEXT
# ============================================================

mean_dewpoint_depression = (
    era5["dewpoint_depression_c"]
    .mean(dim="valid_time")
)

mean_dewpoint_depression.name = (
    "mean_dewpoint_depression"
)

mean_dewpoint_depression.attrs = {
    "long_name": "Mean January dewpoint depression",
    "units": "degC",
}


# Percentage of time where T and dewpoint are within 2 C.
#
# This is NOT an icing criterion.
# It is simply a useful indicator of air frequently
# being close to saturation.
near_saturation_pct = (
    (era5["dewpoint_depression_c"] <= 2)
    .mean(dim="valid_time")
    * 100
)

near_saturation_pct.name = "near_saturation_hours_pct"

near_saturation_pct.attrs = {
    "long_name": (
        "Percentage of January hours with "
        "dewpoint depression <= 2 degC"
    ),
    "units": "%",
}


# ============================================================
# 6. CLOUD METRICS
# ============================================================

mean_low_cloud = (
    era5["low_cloud_cover_pct"]
    .mean(dim="valid_time")
)

mean_low_cloud.name = "mean_low_cloud_cover"

mean_low_cloud.attrs = {
    "long_name": "Mean January low cloud cover",
    "units": "%",
}


low_cloud_50_pct = (
    (era5["low_cloud_cover_pct"] >= 50)
    .mean(dim="valid_time")
    * 100
)

low_cloud_50_pct.name = "low_cloud_ge50_hours_pct"

low_cloud_50_pct.attrs = {
    "long_name": (
        "Percentage of January hours "
        "with at least 50 percent low cloud cover"
    ),
    "units": "%",
}


# Cloud-base height may contain NaN values when a meaningful
# cloud base is unavailable, so use skipna.
median_cloud_base = era5["cbh"].median(
    dim="valid_time",
    skipna=True,
)

median_cloud_base.name = "median_cloud_base_height"

median_cloud_base.attrs = {
    "long_name": "Median January cloud base height",
    "units": era5["cbh"].attrs.get("units", "m"),
}


# ============================================================
# 7. SUPERCOOLED LIQUID WATER
# ============================================================

mean_tcslw = era5["tcslw"].mean(
    dim="valid_time"
)

mean_tcslw.name = "mean_tcslw"

mean_tcslw.attrs = {
    "long_name": (
        "Mean total-column supercooled liquid water"
    ),
    "units": era5["tcslw"].attrs.get(
        "units",
        "kg m-2",
    ),
}


max_tcslw = era5["tcslw"].max(
    dim="valid_time"
)

max_tcslw.name = "max_tcslw"

max_tcslw.attrs = {
    "long_name": (
        "Maximum total-column supercooled liquid water"
    ),
    "units": era5["tcslw"].attrs.get(
        "units",
        "kg m-2",
    ),
}


# Same variable restricted to hours when surface air
# temperature is below freezing.
#
# Still NOT an ice-accretion calculation.
mean_tcslw_subfreezing = (
    era5["tcslw"]
    .where(era5["t2m_c"] < 0)
    .mean(
        dim="valid_time",
        skipna=True,
    )
)

mean_tcslw_subfreezing.name = (
    "mean_tcslw_subfreezing"
)

mean_tcslw_subfreezing.attrs = {
    "long_name": (
        "Mean total-column supercooled liquid water "
        "during subfreezing surface conditions"
    ),
    "units": era5["tcslw"].attrs.get(
        "units",
        "kg m-2",
    ),
}


# ============================================================
# 8. WIND GUST METRICS
# ============================================================

# i10fg = instantaneous modeled 10 m wind gust
max_instant_gust = era5["i10fg"].max(
    dim="valid_time"
)

max_instant_gust.name = "max_instantaneous_gust"

max_instant_gust.attrs = {
    "long_name": (
        "Maximum instantaneous 10 metre wind gust "
        "during January"
    ),
    "units": "m s-1",
}


p95_instant_gust = era5["i10fg"].quantile(
    0.95,
    dim="valid_time",
)

if "quantile" in p95_instant_gust.coords:
    p95_instant_gust = (
        p95_instant_gust.drop_vars("quantile")
    )

p95_instant_gust.name = "p95_instantaneous_gust"

p95_instant_gust.attrs = {
    "long_name": (
        "95th percentile instantaneous "
        "10 metre wind gust"
    ),
    "units": "m s-1",
}


# fg10 = maximum gust since previous post-processing.
#
# Since this is an hourly ERA5 product, this is the more
# useful field for asking:
# "What was the strongest gust captured during this month?"
max_interval_gust = era5["fg10"].max(
    dim="valid_time"
)

max_interval_gust.name = "max_interval_gust"

max_interval_gust.attrs = {
    "long_name": (
        "Maximum 10 metre wind gust since "
        "previous post-processing"
    ),
    "units": "m s-1",
}


p95_interval_gust = era5["fg10"].quantile(
    0.95,
    dim="valid_time",
)

if "quantile" in p95_interval_gust.coords:
    p95_interval_gust = (
        p95_interval_gust.drop_vars("quantile")
    )

p95_interval_gust.name = "p95_interval_gust"

p95_interval_gust.attrs = {
    "long_name": (
        "95th percentile 10 metre interval wind gust"
    ),
    "units": "m s-1",
}


# ============================================================
# 9. PRECIPITATION / SNOWFALL RATE METRICS
# ============================================================

mean_precip_rate = era5[
    "precip_rate_mmh"
].mean(dim="valid_time")

mean_precip_rate.name = "mean_precip_rate"

mean_precip_rate.attrs = {
    "long_name": "Mean January precipitation rate",
    "units": "mm h-1",
}


max_precip_rate = era5[
    "precip_rate_mmh"
].max(dim="valid_time")

max_precip_rate.name = "max_precip_rate"

max_precip_rate.attrs = {
    "long_name": "Maximum January precipitation rate",
    "units": "mm h-1",
}


mean_snowfall_rate = era5[
    "snowfall_rate_mmh"
].mean(dim="valid_time")

mean_snowfall_rate.name = "mean_snowfall_rate"

mean_snowfall_rate.attrs = {
    "long_name": (
        "Mean January snowfall water-equivalent rate"
    ),
    "units": "mm h-1 water equivalent",
}


max_snowfall_rate = era5[
    "snowfall_rate_mmh"
].max(dim="valid_time")

max_snowfall_rate.name = "max_snowfall_rate"

max_snowfall_rate.attrs = {
    "long_name": (
        "Maximum January snowfall water-equivalent rate"
    ),
    "units": "mm h-1 water equivalent",
}


# ============================================================
# 10. WRITE CLEAN COMBINED NETCDF
# ============================================================

era5 = era5.rio.set_spatial_dims(
    x_dim="longitude",
    y_dim="latitude",
)

era5 = era5.rio.write_crs("EPSG:4326")


encoding = {}

for var in era5.data_vars:

    if np.issubdtype(
        era5[var].dtype,
        np.number,
    ):

        encoding[var] = {
            "zlib": True,
            "complevel": 4,
        }


combined_path = (
    OUTPUT_DIR
    / "era5_icing_context_montana_2020_01.nc"
)

era5.to_netcdf(
    combined_path,
    encoding=encoding,
)

print(
    f"\nSaved combined NetCDF: "
    f"{combined_path}"
)


# ============================================================
# 11. EXPORT QGIS-READY TIFFS
# ============================================================

outputs = [

    (
        freezing_pct,
        "jan_2020_era5_freezing_hours_pct.tif",
    ),

    (
        mean_dewpoint_depression,
        "jan_2020_mean_dewpoint_depression_c.tif",
    ),

    (
        near_saturation_pct,
        "jan_2020_near_saturation_hours_pct.tif",
    ),

    (
        mean_low_cloud,
        "jan_2020_mean_low_cloud_cover_pct.tif",
    ),

    (
        low_cloud_50_pct,
        "jan_2020_low_cloud_ge50_hours_pct.tif",
    ),

    (
        median_cloud_base,
        "jan_2020_median_cloud_base_height.tif",
    ),

    (
        mean_tcslw,
        "jan_2020_mean_tcslw_kgm2.tif",
    ),

    (
        max_tcslw,
        "jan_2020_max_tcslw_kgm2.tif",
    ),

    (
        mean_tcslw_subfreezing,
        "jan_2020_mean_tcslw_subfreezing_kgm2.tif",
    ),

    (
        max_instant_gust,
        "jan_2020_max_instantaneous_gust_ms.tif",
    ),

    (
        p95_instant_gust,
        "jan_2020_p95_instantaneous_gust_ms.tif",
    ),

    (
        max_interval_gust,
        "jan_2020_max_interval_gust_ms.tif",
    ),

    (
        p95_interval_gust,
        "jan_2020_p95_interval_gust_ms.tif",
    ),

    (
        mean_precip_rate,
        "jan_2020_mean_precip_rate_mmh.tif",
    ),

    (
        max_precip_rate,
        "jan_2020_max_precip_rate_mmh.tif",
    ),

    (
        mean_snowfall_rate,
        "jan_2020_mean_snowfall_rate_mmh.tif",
    ),

    (
        max_snowfall_rate,
        "jan_2020_max_snowfall_rate_mmh.tif",
    ),
]


for da, filename in outputs:

    export_geotiff(
        da,
        filename,
    )


# ============================================================
# 12. DIAGNOSTICS
# ============================================================

print("\n--- ERA5 ICING-CONTEXT DIAGNOSTICS ---")

print(
    f"Temperature range: "
    f"{float(era5['t2m_c'].min()):.2f} to "
    f"{float(era5['t2m_c'].max()):.2f} °C"
)

print(
    f"Max instantaneous gust: "
    f"{float(era5['i10fg'].max()):.2f} m/s"
)

print(
    f"Max interval gust: "
    f"{float(era5['fg10'].max()):.2f} m/s"
)

print(
    f"Max precipitation rate: "
    f"{float(era5['precip_rate_mmh'].max()):.3f} mm/h"
)

print(
    f"Max snowfall rate: "
    f"{float(era5['snowfall_rate_mmh'].max()):.3f} mm/h SWE"
)

print(
    f"Max TCSLW: "
    f"{float(era5['tcslw'].max()):.4f} "
    f"{era5['tcslw'].attrs.get('units', 'kg m-2')}"
)

print("\nProcessing complete.")

AVG variables: ['avg_tsrwe', 'avg_tprate']
INSTANT variables: ['d2m', 't2m', 'i10fg', 'cbh', 'lcc', 'tcslw']
MAX variables: ['fg10']

Merged dimensions:
Frozen({'valid_time': 744, 'latitude': 21, 'longitude': 29})

Merged variables:
['d2m', 't2m', 'i10fg', 'cbh', 'lcc', 'tcslw', 'avg_tsrwe', 'avg_tprate', 'fg10']

Saved combined NetCDF: derived/era5_icing/era5_icing_context_montana_2020_01.nc
Saved: derived/era5_icing/rasters/jan_2020_era5_freezing_hours_pct.tif
Saved: derived/era5_icing/rasters/jan_2020_mean_dewpoint_depression_c.tif
Saved: derived/era5_icing/rasters/jan_2020_near_saturation_hours_pct.tif
Saved: derived/era5_icing/rasters/jan_2020_mean_low_cloud_cover_pct.tif
Saved: derived/era5_icing/rasters/jan_2020_low_cloud_ge50_hours_pct.tif
Saved: derived/era5_icing/rasters/jan_2020_median_cloud_base_height.tif
Saved: derived/era5_icing/rasters/jan_2020_mean_tcslw_kgm2.tif
Saved: derived/era5_icing/rasters/jan_2020_max_tcslw_kgm2.tif
Saved: derived/era5_icing/rasters/jan_2020_me